In [19]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import MemorySaver

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_tavily import TavilySearch
from langchain_core.tools import tool

import requests
import math
import os

In [14]:
load_dotenv()

True

In [5]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_KEY")
)

In [15]:
search_tool = TavilySearch(
    max_results=5,
    topic = "general",
    search_depth="advanced",
    tavily_api_key=os.getenv("TAVILY_API_KEY")  
)


@tool
def calculator(expression: str) -> str:
    """
    Useful for simple math caclulations.
    Input should be valid math expression.
    Example: 2+2, math.sqrt(16), 10*5
    """
    try:
        allowed = {
            "math": math,
            "abs": abs,
            "round": round,
            "min": min,
            "max": max,
            "sum": sum
        }
        result = eval(expression, {"__builtin__": {}}, allowed)
        return str(result)
    except Exception as e:
        return f"Calculation error: {str(e)}"

@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch the latest stock proce for a given symbol (e.g. AAPL, TSLA)
    using Aplha vantage with API key in the URL
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=FGY6PWNXREZFY603"
    r = requests.get(url)
    return r.json

In [16]:
# Make tool list
tools = [search_tool, calculator, get_stock_price]

llm_with_tools = llm.bind_tools(tools)

In [ ]:
class ChatState(TypedDict):
    message: Annotated[list[BaseMessage], add_messages]

In [20]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_note(state: ChatState):
    "LLM node that may answer or request a tool call"
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}